# Experiment 70-B: Pseudo-Labeling — 갱신된 Pseudo + Unseen 소량 재도입

**실험 이력**:
```
Exp 67 Baseline          OOF 8.7247  LB 10.156
Exp 69 R1 (w_unseen=0.4) OOF 8.6961  LB 10.141  +0.015 개선
Exp 69 R2 (pseudo 갱신)  OOF 8.6991  LB 10.143  역전
Exp 70-A (w_unseen=0.0)  OOF 8.6953  LB 10.133  +0.008 추가 개선
```

**70-B 전략**:
```
확인된 사실:
  ① Unseen pseudo(Exp 67 기반)는 노이즈 → w=0.0이 최선 (70-A 증명)
  ② 70-A 예측은 Exp 67보다 품질이 높은 모델
     → 70-A 예측으로 Unseen pseudo 갱신하면 품질 향상
  ③ 저분산 6개 layout(전략 C)은 global mean 복사 수준 → 계속 제외

70-B 구성:
  pseudo label: Exp 70-A 예측 (submission_70a.csv)
  Seen pseudo : w=1.0 (70-A와 동일)
  Unseen pseudo: 저분산 6개 layout 제외 + w=0.1 / 0.2 그리드 탐색
  → 최적 weight로 10-seed 풀 앙상블
```

In [ ]:
# Cell 1: 패키지 설치 및 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')
!pip install -q lightgbm xgboost catboost
print("설치 완료")

In [ ]:
# Cell 2: Import 및 경로 설정
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import os, sys, warnings, gc

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
warnings.filterwarnings('ignore')

DATA_DIR    = '/content/drive/MyDrive/MyDrive/LightGBM'
TRAIN_PATH  = os.path.join(DATA_DIR, 'train.csv')
TEST_PATH   = os.path.join(DATA_DIR, 'test.csv')
LAYOUT_PATH = os.path.join(DATA_DIR, 'layout_info.csv')

# 70-B pseudo label 출발점: 70-A 예측 (Exp 67보다 품질 높음)
PSEUDO_PRED_PATH = os.path.join(DATA_DIR, 'submission_70a.csv')

# 출력 경로
OUTPUT_SEARCH_PATH = os.path.join(DATA_DIR, 'submission_70b_search.csv')  # weight 탐색용
OUTPUT_70B_PATH    = os.path.join(DATA_DIR, 'submission_70b.csv')          # 최종 제출용

TARGET = 'avg_delay_minutes_next_30m'
print("경로 설정 완료")

In [ ]:
# Cell 3: 유틸 함수 (Exp 67 원본)
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_numeric_dtype(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if   c_min > np.iinfo(np.int8).min  and c_max < np.iinfo(np.int8).max:  df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
                else: df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max: df[col] = df[col].astype(np.float32)
                else: df[col] = df[col].astype(np.float64)
    return df

In [ ]:
# Cell 4: preprocess_data (Exp 67 원본 — 수정 금지)
def preprocess_data(train, test, layout):
    print("Preprocessing...")
    train = train.merge(layout, on='layout_id', how='left')
    test  = test.merge(layout,  on='layout_id', how='left')

    le = LabelEncoder()
    le.fit(pd.concat([train['layout_type'], test['layout_type']]).astype(str))
    train['layout_type'] = le.transform(train['layout_type'].astype(str))
    test['layout_type']  = le.transform(test['layout_type'].astype(str))

    for df in [train, test]:
        df['robot_utilization']    = df['robot_active']       / (df['robot_total']          + 1e-6)
        df['charger_utilization']  = df['robot_charging']     / (df['charger_count']         + 1e-6)
        df['aisle_pressure']       = df['congestion_score']   / (df['aisle_width_avg']        + 1e-6)
        df['intersection_density'] = df['intersection_count'] / (df['floor_area_sqm']         + 1e-6)
        df['pack_station_pressure']= df['order_inflow_15m']   / (df['pack_station_count']     + 1e-6)
        df['bottleneck_risk']      = df['congestion_score'] * df['intersection_density'] / (df['aisle_width_avg'] + 1e-6)
        df['task_intensity']       = df['order_inflow_15m']   / (df['robot_active']           + 1e-6)

    layout_num_cols = ['aisle_width_avg', 'intersection_count', 'robot_total']
    key_ops         = ['congestion_score', 'robot_active', 'order_inflow_15m']
    for l_col in layout_num_cols:
        for o_col in key_ops:
            train[f'{l_col}_x_{o_col}'] = train[l_col] * train[o_col]
            test[f'{l_col}_x_{o_col}']  = test[l_col]  * test[o_col]

    for col in ['congestion_score', 'low_battery_ratio', 'robot_active']:
        train[f'{col}_vel'] = train.groupby('scenario_id')[col].diff(1)
        test[f'{col}_vel']  = test.groupby('scenario_id')[col].diff(1)

    train['time_idx'] = train.groupby('scenario_id').cumcount()
    test['time_idx']  = test.groupby('scenario_id').cumcount()
    train = train.sort_values(['scenario_id', 'time_idx'])
    test  = test.sort_values(['scenario_id', 'time_idx'])

    SEQ_COLS = [
        'order_inflow_15m','unique_sku_15m','robot_active','robot_idle',
        'robot_charging','battery_mean','battery_std','low_battery_ratio',
        'charge_queue_length','avg_charge_wait','congestion_score',
        'max_zone_density','blocked_path_15m','near_collision_15m',
        'fault_count_15m','avg_recovery_time','task_reassign_15m',
        'replenishment_overlap','pack_utilization','loading_dock_util',
        'staging_area_util','label_print_queue'
    ]

    for col in SEQ_COLS:
        first_tr = train.groupby('scenario_id')[col].transform('first')
        train[f'{col}_vs_start']    = train[col] / (first_tr + 1e-6)
        train[f'{col}_delta_start'] = train[col] - first_tr
        first_ts = test.groupby('scenario_id')[col].transform('first')
        test[f'{col}_vs_start']    = test[col] / (first_ts + 1e-6)
        test[f'{col}_delta_start'] = test[col] - first_ts

    for col in SEQ_COLS:
        if col not in train.columns: continue
        for df in [train, test]:
            prev    = df.groupby('scenario_id')[col].shift(1)
            cum_max = prev.groupby(df['scenario_id']).cummax()
            cum_min = prev.groupby(df['scenario_id']).cummin()
            df[f'{col}_vs_cummax'] = df[col] / (cum_max + 1e-6)
            df[f'{col}_vs_cummin'] = df[col] / (cum_min.abs() + 1e-6)
            cum_range = cum_max - cum_min
            df[f'{col}_position_in_range'] = ((df[col] - cum_min) / (cum_range + 1e-3)).clip(0, 2)

    lag_cols = [
        'congestion_score','fault_count_15m','charge_queue_length',
        'low_battery_ratio','blocked_path_15m','avg_recovery_time',
        'near_collision_15m','pack_utilization'
    ]
    for col in lag_cols:
        if col not in train.columns: continue
        for df in [train, test]:
            for lag in [4,5,6,7]: df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)
            prev = df.groupby('scenario_id')[col].shift(1)
            grp  = prev.groupby(df['scenario_id'])
            for w in [7,10,14,20]:
                df[f'{col}_roll{w}_mean'] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
                df[f'{col}_roll{w}_std']  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True)

    SEQ_COLS_BASE = ['order_inflow_15m','unique_sku_15m','robot_active',
                     'low_battery_ratio','charge_queue_length','congestion_score','fault_count_15m']
    remaining = [c for c in SEQ_COLS_BASE if c not in lag_cols]
    for col in remaining:
        for df in [train, test]:
            for lag in [4,5]: df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)
            prev = df.groupby('scenario_id')[col].shift(1)
            grp  = prev.groupby(df['scenario_id'])
            for w in [7,14,20]:
                df[f'{col}_roll{w}_mean'] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
                df[f'{col}_roll{w}_std']  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True)

    for col in SEQ_COLS:
        for df in [train, test]:
            for lag in [8,10,12,15,20,24]:
                df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)

    remaining2 = [c for c in SEQ_COLS if c not in lag_cols]
    for col in remaining2:
        for df in [train, test]:
            prev = df.groupby('scenario_id')[col].shift(1)
            grp  = prev.groupby(df['scenario_id'])
            for w in [14,20]:
                df[f'{col}_roll{w}_mean'] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
                df[f'{col}_roll{w}_std']  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True)

    tr_new = train['scenario_id'].values != np.roll(train['scenario_id'].values, 1); tr_new[0] = True
    ts_new = test['scenario_id'].values  != np.roll(test['scenario_id'].values,  1); ts_new[0]  = True
    for col in SEQ_COLS_BASE:
        for lag in [1,2]:
            tr_l = train[col].shift(lag).values.copy(); ts_l = test[col].shift(lag).values.copy()
            for l in range(lag):
                tr_l[np.roll(tr_new, l)] = np.nan; ts_l[np.roll(ts_new, l)] = np.nan
            train[f'{col}_lag{lag}'] = tr_l; test[f'{col}_lag{lag}'] = ts_l
        train[f'{col}_exp_mean'] = train.groupby('scenario_id')[col].transform(lambda x: x.shift(1).expanding().mean())
        test[f'{col}_exp_mean']  = test.groupby('scenario_id')[col].transform(lambda x: x.shift(1).expanding().mean())

    train['time_ratio'] = train.groupby('scenario_id')['time_idx'].transform(lambda x: x / (x.max() + 1e-6))
    test['time_ratio']  = test.groupby('scenario_id')['time_idx'].transform(lambda x: x / (x.max() + 1e-6))
    for df in [train, test]:
        df['congestion_ratio'] = df['congestion_score'] / (df['congestion_score_exp_mean'] + 1e-6)
        df['steps_remaining']  = df.groupby('scenario_id')['time_idx'].transform('max') - df['time_idx']

    train.fillna(0, inplace=True); test.fillna(0, inplace=True)
    return reduce_mem_usage(train), reduce_mem_usage(test)

In [ ]:
# Cell 5: apply_smoothed_te (Exp 67 원본)
def apply_smoothed_te(df_tr, df_val, target_col, k=30):
    global_mean = df_tr[target_col].mean()
    agg = df_tr.groupby('layout_id')[target_col].agg(['mean','std','median','count']).reset_index()
    agg['layout_mean'] = (agg['count'] * agg['mean'] + k * global_mean) / (agg['count'] + k)
    agg.rename(columns={'std':'layout_std','median':'layout_median','count':'layout_count'}, inplace=True)
    df_val = df_val.merge(agg[['layout_id','layout_mean','layout_std','layout_median','layout_count']], on='layout_id', how='left')
    df_tr  = df_tr.merge( agg[['layout_id','layout_mean','layout_std','layout_median','layout_count']], on='layout_id', how='left')
    df_val['layout_mean']   = df_val['layout_mean'].fillna(global_mean)
    df_val['layout_std']    = df_val['layout_std'].fillna(df_tr[target_col].std())
    df_val['layout_median'] = df_val['layout_median'].fillna(df_tr[target_col].median())
    df_val['layout_count']  = df_val['layout_count'].fillna(0)
    return df_tr, df_val, ['layout_mean','layout_std','layout_median','layout_count']

In [ ]:
# Cell 6: zero_imp_features, 모델 파라미터 (Exp 67 확정값)
zero_imp_features = [
    'charge_queue_length','avg_charge_wait','charge_queue_length_lag2','fault_count_15m_lag2',
    'time_ratio','task_reassign_15m_vs_cummin','low_battery_ratio_vel','low_battery_ratio_lag2',
    'task_reassign_15m','blocked_path_15m_lag8','blocked_path_15m_lag10','near_collision_15m_lag8',
    'near_collision_15m_lag10','fault_count_15m_lag8','fault_count_15m_lag10','avg_recovery_time_lag8',
    'avg_recovery_time_lag10','task_reassign_15m_lag8','task_reassign_15m_lag10','replenishment_overlap_lag8',
    'replenishment_overlap_lag10','robot_charging_lag10','low_battery_ratio_lag8','low_battery_ratio_lag10',
    'charge_queue_length_lag8','charge_queue_length_lag10','avg_charge_wait_lag8','avg_charge_wait_lag10',
    'fault_count_15m_vs_cummax','fault_count_15m_vs_cummin','avg_recovery_time_vs_cummax','task_reassign_15m_vs_cummax',
    'low_battery_ratio_vs_cummax','low_battery_ratio_vs_cummin','charge_queue_length_vs_cummin','avg_charge_wait_vs_cummax',
    'blocked_path_15m_vs_cummax','near_collision_15m_vs_cummin','charge_queue_length_roll14_mean','task_reassign_15m_vs_start',
    'avg_recovery_time_lag5','avg_recovery_time_lag6','avg_recovery_time_lag7','near_collision_15m_lag4',
    'near_collision_15m_lag5','near_collision_15m_lag6','near_collision_15m_lag7','charge_queue_length_roll7_std',
    'charge_queue_length_roll10_mean','low_battery_ratio_lag4','low_battery_ratio_lag5','low_battery_ratio_lag7',
    'blocked_path_15m_lag4','blocked_path_15m_lag5','blocked_path_15m_lag6','blocked_path_15m_lag7',
    'label_print_queue_delta_start','robot_charging_lag15','low_battery_ratio_lag12','low_battery_ratio_lag15',
    'charge_queue_length_lag12','charge_queue_length_lag15','avg_charge_wait_lag12','avg_charge_wait_lag15',
    'congestion_score_lag12','max_zone_density_lag15','blocked_path_15m_lag12','fault_count_15m_lag4',
    'fault_count_15m_lag5','fault_count_15m_lag6','fault_count_15m_lag7','charge_queue_length_lag4',
    'charge_queue_length_lag5','charge_queue_length_lag6','charge_queue_length_lag7','charge_queue_length_roll7_mean',
    'blocked_path_15m_lag15','near_collision_15m_lag12','near_collision_15m_lag15','fault_count_15m_lag12',
    'fault_count_15m_lag15','avg_recovery_time_lag12','avg_recovery_time_lag15','task_reassign_15m_lag12',
    'task_reassign_15m_lag15','replenishment_overlap_lag12','replenishment_overlap_lag15','charge_queue_length_position_in_range',
    'avg_charge_wait_position_in_range','congestion_score_position_in_range','blocked_path_15m_position_in_range',
    'near_collision_15m_position_in_range','fault_count_15m_position_in_range','avg_recovery_time_position_in_range',
    'task_reassign_15m_position_in_range','replenishment_overlap_position_in_range','label_print_queue_position_in_range',
    'replenishment_overlap_vs_cummin','staging_area_util_vs_cummax','battery_mean_position_in_range',
    'low_battery_ratio_position_in_range','label_print_queue_lag15','charge_queue_length_roll20_std',
    'charge_queue_length_vs_start','charge_queue_length_delta_start','avg_charge_wait_vs_start','avg_charge_wait_delta_start'
]

cat_params_base = {
    'iterations': 1441, 'learning_rate': 0.024382726628741795, 'depth': 7,
    'l2_leaf_reg': 4.329713228202991, 'bagging_temperature': 0.15517607913494932,
    'loss_function': 'MAE', 'eval_metric': 'MAE', 'verbose': False,
    'task_type': 'GPU', 'devices': '0', 'early_stopping_rounds': 100
}

gkf   = GroupKFold(n_splits=5)
seeds = [42, 123, 2026, 777, 1004, 314, 555, 888, 999, 1337]

def target_forward(y): return np.log1p(y)
def target_inverse(p): return np.expm1(p)

print("파라미터 설정 완료")

In [ ]:
# Cell 7: 데이터 로드 및 전처리
print("--- Experiment 70-B ---")
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
layout    = pd.read_csv(LAYOUT_PATH)

train_layouts  = set(train_raw['layout_id'].unique())
test_layouts   = set(test_raw['layout_id'].unique())
seen_layouts   = train_layouts & test_layouts
unseen_layouts = test_layouts - train_layouts

print(f"Seen layouts  : {len(seen_layouts)}개 | Unseen layouts: {len(unseen_layouts)}개")

train, test = preprocess_data(train_raw, test_raw, layout)

features_base   = [c for c in train.columns
                   if c not in ['ID','layout_id','scenario_id',TARGET,'is_seen','is_pseudo']]
features_pruned = [f for f in features_base if f not in zero_imp_features]

train['is_seen']  = train['layout_id'].isin(seen_layouts)
test['is_seen']   = test['layout_id'].isin(seen_layouts)
test['is_pseudo'] = False

print(f"Train: {train.shape} | Test: {test.shape} | 피처: {len(features_pruned)}개")

In [ ]:
# Cell 8: Pseudo Label 로드 및 품질 분석
#
# [70-B pseudo label 변경 이유]
# Exp 67 예측(Round 0) → 70-A 예측으로 교체
#   70-A는 Seen pseudo를 학습에 포함한 더 나은 모델
#   → Unseen layout에 대한 예측 품질도 향상됨
#   → 갱신된 pseudo로 다시 Unseen을 소량 재도입 시도

if not os.path.exists(PSEUDO_PRED_PATH):
    raise FileNotFoundError(f"{PSEUDO_PRED_PATH} 없음. submission_70a.csv를 확인하세요.")

pseudo_df = pd.read_csv(PSEUDO_PRED_PATH)
test = test.merge(pseudo_df.rename(columns={TARGET: 'pseudo_label'}), on='ID', how='left')

# 70-A 예측과 Exp 67 예측의 차이 확인 (수렴 여부)
print("[70-A pseudo label 통계]")
print(f"  Seen   mean={test[test['is_seen']]['pseudo_label'].mean():.3f}  "
      f"std={test[test['is_seen']]['pseudo_label'].std():.3f}  "
      f"({test['is_seen'].sum()}행)")
print(f"  Unseen mean={test[~test['is_seen']]['pseudo_label'].mean():.3f}  "
      f"std={test[~test['is_seen']]['pseudo_label'].std():.3f}  "
      f"({(~test['is_seen']).sum()}행)")

# 저분산 Unseen layout 탐지 (Exp 69와 동일 기준)
train_std      = train[TARGET].std()
train_q01      = train[TARGET].quantile(0.01)
train_q99      = train[TARGET].quantile(0.99)

unseen_test        = test[~test['is_seen']]
layout_std_map     = unseen_test.groupby('layout_id')['pseudo_label'].std()
low_var_layouts    = layout_std_map[layout_std_map < train_std * 0.3].index.tolist()
mask_low_var       = test['layout_id'].isin(low_var_layouts) & ~test['is_seen']
mask_extreme       = (test['pseudo_label'] < train_q01) | (test['pseudo_label'] > train_q99)

print(f"\n[Unseen layout 분석 (70-A 예측 기준)]")
print(f"  전체 Unseen layouts   : {len(layout_std_map)}개")
print(f"  저분산 layouts        : {len(low_var_layouts)}개  "
      f"(std < {train_std*0.3:.3f})")
print(f"  저분산 해당 행        : {mask_low_var.sum()}행")
print(f"  극단값 행             : {mask_extreme.sum()}행")
print()
print(f"  → 70-B에서 사용할 Unseen pseudo: "
      f"{(~mask_low_var & ~mask_extreme & ~test['is_seen']).sum()}행 "
      f"(저분산·극단값 제외)")

In [ ]:
# Cell 9: 통합 데이터셋 구성 함수

def make_sample_weight(df, w_orig=1.0, w_seen=1.0, w_unseen=0.1):
    """
    w_orig  : 원본 train 행 weight
    w_seen  : Pseudo Seen 행 weight  (70-A에서 1.0이 최적 확인)
    w_unseen: Pseudo Unseen 행 weight (탐색 대상: 0.0 / 0.1 / 0.2)
    """
    is_pseudo = df['is_pseudo'].fillna(False).values
    is_seen   = df['is_seen'].values
    weights = np.where(~is_pseudo, w_orig,
              np.where(is_seen,    w_seen, w_unseen))
    return weights

def build_combined(train_orig, pseudo_set, pseudo_label_col='pseudo_label'):
    ps = pseudo_set.copy()
    ps[TARGET]      = ps[pseudo_label_col]
    ps['is_pseudo'] = True
    shared_cols = [c for c in train_orig.columns if c in ps.columns]
    combined = pd.concat([train_orig[shared_cols], ps[shared_cols]], ignore_index=True)
    combined['is_pseudo'] = combined['is_pseudo'].fillna(False)
    return combined

# 전략 C 기반 pseudo 셋 구성
# Seen: 전량 사용 (극단값만 제거)
# Unseen: 저분산 6개 layout + 극단값 제거 (전략 C)
pseudo_seen_mask   = test['is_seen'] & ~mask_extreme
pseudo_unseen_mask = ~test['is_seen'] & ~mask_low_var & ~mask_extreme

pseudo_seen_rows   = test[pseudo_seen_mask]
pseudo_unseen_rows = test[pseudo_unseen_mask]
pseudo_all_C       = test[pseudo_seen_mask | pseudo_unseen_mask]

print(f"[70-B 통합 데이터셋 구성]")
print(f"  원본 train          : {len(train):>6}행  weight=1.0")
print(f"  Pseudo Seen         : {len(pseudo_seen_rows):>6}행  weight=1.0")
print(f"  Pseudo Unseen (전략C): {len(pseudo_unseen_rows):>6}행  weight=탐색 (0.0/0.1/0.2)")
print(f"  Total               : {len(train)+len(pseudo_all_C):>6}행")

In [ ]:
# Cell 10: [Stage 1] w_unseen 그리드 탐색
#
# seed=42 단독, LGB만으로 빠르게 탐색
# 탐색 대상: w_unseen ∈ {0.0, 0.1, 0.2}
#   0.0 → 70-A와 동일 (Seen only 기준선)
#   0.1 → 소량 재도입
#   0.2 → 적당한 재도입
#
# 소요 시간: 약 15~20분

print("[Stage 1] w_unseen 그리드 탐색 (LGB seed=42 단독)")
print("  탐색: w_unseen ∈ {0.0, 0.1, 0.2}\n")

train_combined = build_combined(train, pseudo_all_C)
orig_mask      = ~train_combined['is_pseudo'].fillna(False).values
y_orig         = train_combined[TARGET].values[orig_mask]

search_results = {}

for w_unseen in [0.0, 0.1, 0.2]:
    print(f"\n  w_unseen={w_unseen}")
    oof_w = np.zeros(len(train_combined))

    for fold, (tr_idx, val_idx) in enumerate(
            gkf.split(train_combined, train_combined[TARGET],
                      groups=train_combined['layout_id'])):

        X_tr  = train_combined.iloc[tr_idx].copy()
        y_tr  = train_combined.iloc[tr_idx][TARGET]
        # w_seen=1.0 고정, w_unseen만 변화
        sw_tr = make_sample_weight(train_combined.iloc[tr_idx],
                                   w_orig=1.0, w_seen=1.0, w_unseen=w_unseen)
        X_val = train_combined.iloc[val_idx].copy()
        y_val = train_combined.iloc[val_idx][TARGET]

        X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
        X_tr.fillna(0, inplace=True); X_val.fillna(0, inplace=True)
        feats = features_pruned + te_cols

        m = LGBMRegressor(
            learning_rate=0.03, n_estimators=1500, max_depth=7, num_leaves=127,
            verbose=-1, objective='huber', subsample=0.75, colsample_bytree=0.65,
            subsample_freq=1, random_state=42
        )
        m.fit(
            X_tr[feats], target_forward(y_tr),
            sample_weight=sw_tr,
            eval_set=[(X_val[feats], target_forward(y_val))],
            callbacks=[lgb.early_stopping(100)]
        )
        oof_w[val_idx] = target_inverse(m.predict(X_val[feats]))
        print(f"    Fold {fold+1}", end=' | ', flush=True)

    oof_w = np.maximum(oof_w, 0)
    mae   = mean_absolute_error(y_orig, oof_w[orig_mask])
    search_results[w_unseen] = mae
    print(f"\n  → OOF MAE(orig): {mae:.4f}")
    gc.collect()

# 탐색 결과 요약
print("\n" + "="*50)
print("[Stage 1 결과 요약]")
best_w_unseen = min(search_results, key=search_results.get)
for w, mae in sorted(search_results.items()):
    marker = " ← 최적" if w == best_w_unseen else ""
    print(f"  w_unseen={w:.1f}  OOF: {mae:.4f}{marker}")
print(f"\n→ 최적 w_unseen={best_w_unseen} 로 Stage 2 풀 앙상블 진행")
print("="*50)

In [ ]:
# Cell 11: [Stage 2] 최적 w_unseen으로 10-Seed 풀 앙상블
#
# Stage 1에서 결정된 best_w_unseen 사용
# w_unseen=0.0이 최적이면 → 70-A와 동일하지만 pseudo label이 갱신됨
# w_unseen>0.0이 최적이면 → Unseen 재도입 효과 확인

print(f"[Stage 2] 10-Seed 풀 앙상블 (w_unseen={best_w_unseen})")
print(f"  학습 데이터: {len(train_combined)}행")
print(f"  Seen pseudo weight  : 1.0")
print(f"  Unseen pseudo weight: {best_w_unseen}")
print(f"  약 2시간 소요\n")

oof_70b  = np.zeros(orig_mask.sum())
test_70b = np.zeros(len(test))

for seed_idx, seed in enumerate(seeds):
    print(f"\n{'='*28} Seed {seed} ({seed_idx+1}/10) {'='*28}")

    oof_lgb = np.zeros(len(train_combined))
    oof_xgb = np.zeros(len(train_combined))
    oof_cat = np.zeros(len(train_combined))
    t_lgb   = np.zeros(len(test))
    t_xgb   = np.zeros(len(test))
    t_cat   = np.zeros(len(test))

    for fold, (tr_idx, val_idx) in enumerate(
            gkf.split(train_combined, train_combined[TARGET],
                      groups=train_combined['layout_id'])):

        X_tr  = train_combined.iloc[tr_idx].copy()
        y_tr  = train_combined.iloc[tr_idx][TARGET]
        sw_tr = make_sample_weight(train_combined.iloc[tr_idx],
                                   w_orig=1.0, w_seen=1.0, w_unseen=best_w_unseen)
        X_val = train_combined.iloc[val_idx].copy()
        y_val = train_combined.iloc[val_idx][TARGET]

        X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
        X_tr.fillna(0, inplace=True); X_val.fillna(0, inplace=True)
        _, X_te, _ = apply_smoothed_te(X_tr, test.copy(), TARGET, k=30)
        X_te.fillna(0, inplace=True)
        feats   = features_pruned + te_cols
        y_tr_t  = target_forward(y_tr)
        y_val_t = target_forward(y_val)

        # LightGBM
        m_lgb = LGBMRegressor(
            learning_rate=0.03, n_estimators=1500, max_depth=7, num_leaves=127,
            verbose=-1, objective='huber', subsample=0.75, colsample_bytree=0.65,
            subsample_freq=1, random_state=seed
        )
        m_lgb.fit(
            X_tr[feats], y_tr_t, sample_weight=sw_tr,
            eval_set=[(X_val[feats], y_val_t)],
            callbacks=[lgb.early_stopping(100)]
        )
        oof_lgb[val_idx] = target_inverse(m_lgb.predict(X_val[feats]))
        t_lgb += target_inverse(m_lgb.predict(X_te[feats])) / 5

        # XGBoost
        m_xgb = XGBRegressor(
            learning_rate=0.03, n_estimators=1000, max_depth=7,
            objective='reg:absoluteerror', verbosity=0,
            early_stopping_rounds=100, subsample=0.75,
            colsample_bytree=0.65, random_state=seed, device='cuda'
        )
        m_xgb.fit(
            X_tr[feats], y_tr_t, sample_weight=sw_tr,
            eval_set=[(X_val[feats], y_val_t)], verbose=False
        )
        oof_xgb[val_idx] = target_inverse(m_xgb.predict(X_val[feats]))
        t_xgb += target_inverse(m_xgb.predict(X_te[feats])) / 5

        # CatBoost
        m_cat = CatBoostRegressor(**{**cat_params_base, 'random_seed': seed})
        m_cat.fit(
            X_tr[feats], y_tr_t, sample_weight=sw_tr,
            eval_set=[(X_val[feats], y_val_t)]
        )
        oof_cat[val_idx] = target_inverse(m_cat.predict(X_val[feats]))
        t_cat += target_inverse(m_cat.predict(X_te[feats])) / 5

        print(f"Fold {fold+1}", end=' | ', flush=True)

    oof_lgb = np.maximum(oof_lgb, 0)
    oof_xgb = np.maximum(oof_xgb, 0)
    oof_cat = np.maximum(oof_cat, 0)

    mae_lgb = mean_absolute_error(y_orig, oof_lgb[orig_mask])
    mae_xgb = mean_absolute_error(y_orig, oof_xgb[orig_mask])
    mae_cat = mean_absolute_error(y_orig, oof_cat[orig_mask])
    inv = np.array([1/mae_lgb, 1/mae_xgb, 1/mae_cat])
    w   = inv / inv.sum()

    seed_oof  = (oof_lgb*w[0] + oof_xgb*w[1] + oof_cat*w[2])[orig_mask]
    seed_test =  t_lgb*w[0]   + t_xgb*w[1]   + t_cat*w[2]

    oof_70b  += seed_oof  / len(seeds)
    test_70b += seed_test / len(seeds)

    print(f"\nSeed {seed} OOF: {mean_absolute_error(y_orig, seed_oof):.4f} "
          f"(LGB:{mae_lgb:.4f} XGB:{mae_xgb:.4f} CAT:{mae_cat:.4f})")
    gc.collect()

test_70b = np.maximum(test_70b, 0)

In [ ]:
# Cell 12: 결과 저장 및 최종 요약
mae_70b        = mean_absolute_error(train[TARGET], oof_70b)
seen_mae_70b   = mean_absolute_error(
    train[TARGET].values[train['is_seen'].values],
    oof_70b[train['is_seen'].values]
)
unseen_mae_70b = mean_absolute_error(
    train[TARGET].values[~train['is_seen'].values],
    oof_70b[~train['is_seen'].values]
)

pd.DataFrame({'ID': test['ID'], TARGET: test_70b}).to_csv(OUTPUT_70B_PATH, index=False)

print("\n" + "="*68)
print("  [Exp 70-B 최종 결과]")
print("="*68)
print(f"  {'실험':<35} {'OOF MAE':>10}  {'LB':>8}")
print("-"*68)
print(f"  {'Exp 67 Baseline':<35} {'8.7247':>10}  {'10.156':>8}")
print(f"  {'Exp 69 R1 (w_unseen=0.4)':<35} {'8.6961':>10}  {'10.141':>8}")
print(f"  {'Exp 70-A (w_unseen=0.0)':<35} {'8.6953':>10}  {'10.133':>8}")
print(f"  {'Exp 70-B (w_unseen=' + str(best_w_unseen) + ', 갱신 pseudo)':<35} {mae_70b:>10.4f}  {'제출 전':>8}")
print("="*68)
print(f"  Seen   MAE : {seen_mae_70b:.4f}")
print(f"  Unseen MAE : {unseen_mae_70b:.4f}")
print(f"  저장       : {OUTPUT_70B_PATH}")
print("="*68)
print()
print("[Stage 1 w_unseen 탐색 결과]")
for w, mae in sorted(search_results.items()):
    marker = " ← 채택" if w == best_w_unseen else ""
    print(f"  w_unseen={w:.1f}  OOF: {mae:.4f}{marker}")
print()
print("[LB 제출 후 다음 방향]")
print("  70-B LB < 10.133 → pseudo 갱신 + 전략 C 효과 확인. Round 2 진행 가능")
print("  70-B LB > 10.133 → 70-A가 현재 최선. 피처 방향으로 전환 고려")